# Train ABSA Deep Learning Model on PC

This notebook trains the deep learning model for the restaurant ABSA project on a Windows/NVIDIA PC.

It uses the standalone training script in the same folder:

`train_on_pc.py`

The notebook is designed to be opened from this folder:

`backends/deep_learning/training_on_pc/`

Important: do not replace the existing project model until the new report is better than the old result.

## 1. Check Files

This cell checks that the notebook is running in the correct folder and that the required dataset/training files exist.

In [ ]:
from pathlib import Path

WORK_DIR = Path.cwd()
TRAIN_SCRIPT = WORK_DIR / "train_on_pc.py"
DATASET_PATH = WORK_DIR / "restaurant_absa_aspect_rows.csv"
WIDE_DATASET_PATH = WORK_DIR / "restaurant_absa_4_aspects.csv"
OUTPUT_DIR = WORK_DIR / "model_output" / "bert_absa_model"
REPORT_PATH = WORK_DIR / "model_output" / "bert_training_report.json"

print("Working folder:", WORK_DIR)
print("Training script exists:", TRAIN_SCRIPT.exists(), TRAIN_SCRIPT)
print("Long dataset exists:", DATASET_PATH.exists(), DATASET_PATH)
print("Wide dataset exists:", WIDE_DATASET_PATH.exists(), WIDE_DATASET_PATH)

assert TRAIN_SCRIPT.exists(), "Missing train_on_pc.py. Open this notebook inside the training_on_pc folder."
assert DATASET_PATH.exists(), "Missing restaurant_absa_aspect_rows.csv."
assert WIDE_DATASET_PATH.exists(), "Missing restaurant_absa_4_aspects.csv."

## 2. Inspect Dataset Distribution

This cell shows the label imbalance before training. The `Unknown` class is much larger than `Positive` and `Negative`, so macro F1 is more important than accuracy alone.

In [ ]:
import pandas as pd

df = pd.read_csv(DATASET_PATH, dtype={"id": str})
print("Dataset shape:", df.shape)
print("Unique reviews:", df["id"].nunique())
print("\nLabel distribution:")
display(df["sentiment"].value_counts().to_frame("count"))

print("\nLabel distribution by aspect:")
aspect_label_counts = (
    df.groupby(["aspect", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Positive", "Negative", "Unknown"], fill_value=0)
)
display(aspect_label_counts)

## 3. Choose Training Settings

Start with the safe baseline settings below. This should reproduce a normal model first.

Recommended first run:

- `class_weighted_loss = False`
- `max_unknown_to_known_ratio = 0.0`
- `learning_rate = "2e-5"`
- `epochs = 5`

Only try class-weighted loss after the baseline works. Do not combine class-weighted loss with downsampling unless you are experimenting carefully.

In [ ]:
# Safe baseline configuration
model_name = "microsoft/deberta-v3-base"
epochs = 5
batch_size = 8
eval_batch_size = 16
gradient_accumulation_steps = 2
learning_rate = "2e-5"
max_length = 160

# Keep these disabled for the first run.
class_weighted_loss = False
max_unknown_to_known_ratio = 0.0

# Set this to True if FP16 training becomes unstable on your PC.
no_fp16 = False

# Set this to True for a quick test only. Do not use quick mode for final results.
quick = False
quick_rows = 1000

print("Training configuration ready.")

## 4. Build Training Command

This cell builds the command that will run `train_on_pc.py` with the settings above.

In [ ]:
import sys

cmd = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--model", model_name,
    "--epochs", str(epochs),
    "--batch-size", str(batch_size),
    "--eval-batch-size", str(eval_batch_size),
    "--gradient-accumulation-steps", str(gradient_accumulation_steps),
    "--learning-rate", str(learning_rate),
    "--max-length", str(max_length),
]

if class_weighted_loss:
    cmd.append("--class-weighted-loss")

if max_unknown_to_known_ratio > 0:
    cmd.extend(["--max-unknown-to-known-ratio", str(max_unknown_to_known_ratio)])

if no_fp16:
    cmd.append("--no-fp16")

if quick:
    cmd.extend(["--quick", "--quick-rows", str(quick_rows)])

print("Command:")
print(" ".join(cmd))

## 5. Train Model

Run this cell to start training. It may take a long time depending on your GPU.

The script will save:

- `model_output/bert_absa_model/`
- `model_output/bert_training_report.json`
- `model_output/bert_absa_model.zip`

In [ ]:
import subprocess

result = subprocess.run(cmd, cwd=WORK_DIR)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

print("Training finished.")

## 6. Read Training Report

After training, this cell reads the JSON report and prints the important numbers.

Compare the new result with the old model:

- Old accuracy: `0.9197`
- Old macro F1: `0.8205`
- Old weighted F1: `0.9206`
- Old Positive F1: `0.7829`
- Old Negative F1: `0.7227`
- Old Unknown F1: `0.9559`

Only use the new model if it is close to or better than these values, especially macro F1.

In [ ]:
import json

assert REPORT_PATH.exists(), f"Report not found: {REPORT_PATH}"

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

print("Rows:")
print(json.dumps(report["rows"], indent=2))

print("\nTraining args:")
print(json.dumps(report["training_args"], indent=2))

print("\nTest metrics:")
for key, value in report["test_metrics"].items():
    print(f"{key}: {value:.4f}")

print("\nPer-label metrics:")
display(pd.DataFrame(report["per_label"]).T)

print("\nConfusion matrix labels:", report["confusion_matrix_labels"])
display(pd.DataFrame(
    report["confusion_matrix"],
    index=["true_" + label for label in report["confusion_matrix_labels"]],
    columns=["pred_" + label for label in report["confusion_matrix_labels"]],
))

## 7. Test A Few Demo Sentences

If the report looks good, test the saved model on some demo sentences.

This cell loads the model from `model_output/bert_absa_model/` and predicts the four project aspects.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ASPECTS = [
    "Food",
    "Service",
    "Price",
    "Eating Environment / Ambiance",
]

ID2LABEL = {0: "Positive", 1: "Negative", 2: "Unknown"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR, local_files_only=True)
model.to(device)
model.eval()

def predict_review(review: str):
    encoded = tokenizer(
        [review] * len(ASPECTS),
        ASPECTS,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = model(**encoded)
        probabilities = torch.softmax(output.logits, dim=-1).cpu()
        predictions = probabilities.argmax(dim=-1).tolist()

    rows = []
    for aspect, label_id, probs in zip(ASPECTS, predictions, probabilities):
        rows.append({
            "aspect": aspect,
            "sentiment": ID2LABEL[int(label_id)],
            "confidence": round(float(probs[label_id]), 4),
        })
    return pd.DataFrame(rows)

demo_review = "The food was tasty, but the waiter was slow and the atmosphere was noisy."
print(demo_review)
display(predict_review(demo_review))

## 8. What To Copy Back To The Main Project

If the new model is better, copy these files/folders back to the Mac project:

From PC:

- `training_on_pc/model_output/bert_absa_model/`
- `training_on_pc/model_output/bert_training_report.json`

To Mac project:

- `backends/deep_learning/bert_absa_model/`
- `backends/deep_learning/bert_training_report.json`

If the new result is worse, keep the old model.

## Optional Experiments

After the safe baseline works, you can try one controlled improvement at a time.

Experiment A: class-weighted loss only

```python
class_weighted_loss = True
max_unknown_to_known_ratio = 0.0
learning_rate = "1e-5"
epochs = 5
```

Experiment B: light Unknown downsampling only

```python
class_weighted_loss = False
max_unknown_to_known_ratio = 3.0
learning_rate = "2e-5"
epochs = 5
```

Avoid combining class-weighted loss and aggressive downsampling unless the separate experiments work well.